In [2]:
import pandas as pd
import numpy as np
from pypfopt.expected_returns import mean_historical_return
from pypfopt.discrete_allocation import DiscreteAllocation
from collections import OrderedDict
from pandas import DataFrame
from pandas import Series
from typing import Any
from numpy.typing import NDArray
from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.efficient_frontier import EfficientFrontier
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
from pydantic import BaseModel
from typing import TypedDict
import yfinance as yf

import time
import json
from abc import ABC, abstractmethod

import bidask as ba

In [ ]:


class PricesData(ABC):
    @abstractmethod
    def get_data(self):
        pass

class ApiOrMockPricesData(PricesData):
    def __init__(self,assets_tickers:list[str],start_date:str,interval:str):
        self.is_default=False
        self.assets_tickers=assets_tickers
        self.start_date=start_date
        self.interval=interval
    
    def safe_download(self,assets_tickers:list[str],start_date:str)->pd.DataFrame|None:
        try:

            
            cleaned_prices=clean_data( yf.download(
                assets_tickers,
                period='max',
                auto_adjust=False,
                threads=True,
                interval=self.interval,
                group_by="ticker"
            ),0.05,3)
            

            return cleaned_prices if cleaned_prices is not None else None

        except Exception as e:
            print(f"[ERROR] Download failed: {e}")
            return None

    def robust_download(self, max_retries=5)->pd.DataFrame|None:
        for attempt in range(max_retries):
            try:
                cleaned_prices:pd.DataFrame = self.safe_download(self.assets_tickers,self.start_date)#third pass of the start_date variable
                if cleaned_prices is not None and not cleaned_prices.empty:
                    return cleaned_prices
            except Exception as e:
                print(f"[Retry {attempt+1}] {e}")     
            time.sleep(2)  # wait before retry
            
        print("[FAILURE] All retries failed")
        return None

    def get_data(self)->tuple[bool,pd.DataFrame]:
        cleaned_prices:pd.DataFrame = self.robust_download()
        if cleaned_prices is None:
            print("[WARNING] Using fallback data")
            self.is_default=True
            cleaned_prices=pd.read_parquet("cleaned_multiindex_prices.parquet")
        return self.is_default,cleaned_prices
    
class MockData(PricesData):
    def __init__(self,file_name:str):
        self.is_default=True
        self.file_name=file_name

    def get_data(self)->tuple[bool,pd.DataFrame]:
        return self.is_default,pd.read_parquet(self.file_name)

class HistoricalPricesService:
    """ 
        This class uses the DIP design pattern; the domain logic does not depend on hardcoded input, but depend on interface.
        In other words, this class depend on static and non-volatile entity, which is the interface in this case.
        In other words, higher level policy depends does not depend on lower level policy.
        The higher level policy-the HistoricalPricesService- uses/controls the lower level policy and does not depend on it.
        The lower level policy does depend on a higher level policy-the interface- by implmenting the abstract methods defined inside that interface. 
    """
    def __init__(self, data_source: PricesData):
        self.data_source = data_source
        
    def get_data(self)->tuple[bool,pd.DataFrame]:
        return self.data_source.get_data()


def clean_data(df:pd.DataFrame, nan_percentage:float,fill_max_gap:int=4)->pd.DataFrame:
    """
        this function perform three cleaning phases on a dataframe:
        1- if any column has a lot of missing values, the function will drop this column entirely, if the number of missing values is acceptable, the function drops the records that contains missing data without deleting the whole column.

    Args:
        df (pd.DataFrame): Two Diminsional DataFrame.
        nan_percentage (float): Upper bound for the amount of  missing values.

    Returns:
        pd.DataFrame: table that contains no missing values.
    """
    old_num_rows=df.shape[0]
    print("number of rows, which are prices records, before cleaning is :",old_num_rows)
    
    cols = [col for col in df.columns if (df[col].isna().sum()/df[col].shape[0])>nan_percentage]
    print("\ndeleted columns are : ",cols,"\n")
    df = df.drop(cols, axis=1)
    df.ffill(limit=fill_max_gap,inplace=True)
    df.dropna(inplace=True)
    
    print("numnber of  prices records after cleaning: ",df.shape[0],'\n')
    print(100*df.shape[0]/old_num_rows,"%","of the prices records survived the cleaning, the more the better to keep the observations to be daily observations as possible  \n")
    return df


class ReturnsAndPrices(TypedDict):
    selectedAssetsReturns:pd.DataFrame
    selectedAssetsPrices:pd.DataFrame
 


class EfficientFrontierPointStaticType(TypedDict):
    volatility: float
    expectedReturn: float

def compute_frontier(
        ef:EfficientFrontier, 
        points=50)->list[EfficientFrontierPointStaticType]:
    """determine the volatality and expected return for each efficient portfolio

    Args:
        ef (_type_): _description_
        points (int, optional): _description_. Defaults to 50.

    Returns:
        list[EfficientFrontierPointStaticType]: _description_
    """
    import numpy as np

    ef_min = ef.deepcopy()
    ef_max = ef.deepcopy()

    ef_min.min_volatility()
    min_ret = ef_min.portfolio_performance()[0]

    max_ret = ef_max._max_return()

    ef_range = np.linspace(min_ret, max_ret - 1e-4, points)

    frontier = []

    for target in ef_range:
        ef_copy = ef.deepcopy()
        try:
            ef_copy.efficient_return(target)
            ret, vol, _ = ef_copy.portfolio_performance()

            frontier.append({
                "volatility": float(vol),
                "expectedReturn": float(ret)
            })

        except Exception:
            continue

    
    frontier.sort(key=lambda x: x["volatility"])

    return frontier

class AssetScatterPointStaticType(TypedDict):
      ticker: str
      volatility: float
      expectedReturn:float

class PerformenceMetricsStaticType(TypedDict):
        expectedAnnualReturn: float
        annualVolatility: float   
        sharpeRatio: float
        
class AssetsAllocationsResults(TypedDict):
    leftover:float
    sharesQuantities:dict[str,int]
    riskReturnScatterPoints:list[AssetScatterPointStaticType]
    efficientFrontierPoints:list[EfficientFrontierPointStaticType]
    capitalAllocationsPercentages:dict[str,float]
    performanceMetrics:PerformenceMetricsStaticType








class Asset(BaseModel):
    assetName: str
    capitalAllocationPercentage: float
    quantity: int
          
class Metrics(BaseModel): 

    expectedAnnualReturn: float
    annualVolatility: float
    sharpeRatio: float

class OptimalPortfolio(BaseModel): 

    assets:list[Asset] 
    metrics:Metrics 
      
class AssetScatterPoint(BaseModel):
    ticker: str
    volatility: float
    expectedReturn:float
      
class EfficientFrontierPoint(BaseModel): 
    volatility: float
    expectedReturn: float

class  InvestementsAdviceMocks (BaseModel) :
      leftover: float
      optimalPortfolio:OptimalPortfolio 
      assetsScatter: list[AssetScatterPoint]
      efficientFrontierPoints: list[EfficientFrontierPoint]
class InvestementsAdviceOrchestrator:# why not to use paranthesis  like (BaseModel) or other stuff ?
    #constructor
    def __init__(self,questions_scores:list[int],answers_weights:list[int], total_portfolio_value:float):
        self.questions_scores=questions_scores
        self.answers_weights=answers_weights
        self.total_portfolio_value=total_portfolio_value


    def get_allocations_percentages(self)->OrderedDict[int,float]:
            

        #self.ef.add_objective(objective_functions.L2_reg, gamma=1)
        if self.risk_appetite=="Conservative":
            
            #class_lower = {"Commodity": 0,"Fixed Income":0.35,"Equities":0.25}
            #class_upper = {"Commodity": 0.20,"Fixed Income":0.70,"Equities":0.55}
            #self.ef.add_sector_constraints(class_mapper, class_lower, class_upper)  
            capital_allocations_percentages:OrderedDict[int, float]=self.ef.min_volatility()
            print("optimized for lowest risk")     
        
        elif self.risk_appetite=="Moderate":
            #class_lower = {"Commodity": 0.0,"Fixed Income":0.20,"Equities":0.45}
            #class_upper = {"Commodity": 0.15,"Fixed Income":0.50,"Equities":0.75}
            #self.ef.add_sector_constraints(class_mapper, class_lower, class_upper)  

            capital_allocations_percentages:OrderedDict[int,float] = self.ef.max_sharpe()
            print("optimzed for maximum sharp ratio")
        else:#aggressive 
            
            ef_copy:EfficientFrontier = self.ef.deepcopy()
            ef_copy.min_volatility()
            min_risk:float = ef_copy.portfolio_performance()[1]
            max_risk = float(np.max(self.assets_volatilities))
            #class_lower = {"Commodity": 0.0,"Fixed Income":0.0,"Equities":0.65}
            #class_upper = {"Commodity": 0.10,"Fixed Income":0.30,"Equities":0.90}        
            #self.ef.add_sector_constraints(class_mapper, class_lower, class_upper)

            capital_allocations_percentages:OrderedDict[int,float] = self.ef.efficient_risk(target_volatility=max(min_risk, min(self.normalized_score,max_risk)))
            print("optimized for maximum return given target risk (aggressive)")
        return capital_allocations_percentages 

    def perform_assets_allocation(self)->None:
        
        returns:pd.DataFrame = self.cleaned_prices.xs("Close",level=1,axis=1).pct_change().iloc[1:]  
        
        covarience_matrix :DataFrame | NDArray[Any] | Any= CovarianceShrinkage(returns,returns_data=True).ledoit_wolf()       
        self.assets_volatilities:NDArray[Any] = np.sqrt(np.diag(covarience_matrix))                               
        
        #market_prices = self.cleaned_prices.xs("Close", level=1, axis=1).mean(axis=1) # Proxy for market portfolio if SPY missing
        #delta = black_litterman.market_implied_risk_aversion(market_prices)
        # We need market caps for all selected assets to calculate the prior
        # For simplicity in this step, if an asset lacks a market cap, we assign a median value to keep the math stable
        #note to Gemini: equities_market_caps includes the caps only for the equities, not all the assets classes. I have more than just equities.
        #mcaps = self.equities_market_caps.reindex(returns.columns).fillna(self.equities_market_caps.median())
        
        #prior_returns = black_litterman.market_implied_prior_returns(mcaps, delta, cov_matrix)

        # Initialize Black Litterman Model (Empty views defaults to using just the market prior, which is still highly stabilizing)
        #bl = BlackLittermanModel(cov_matrix, pi=prior_returns)
        #bl_posterior_rets = bl.bl_returns()
        #bl_posterior_cov = bl.bl_cov()
        #tickers = list(bl_posterior_rets.index)
        annualized_mean_returns :  (Series | Any)= mean_historical_return(returns,returns_data=True,frequency=252) # PPROBLEM : use Series[Any] for mypy, python does not accept it . question to chatGPT : my mind tell me to use black-letterman knowledge to refine the returns, is that scinetific ? I do not even understand what is that.
        tickers:list[str] = list(annualized_mean_returns.index)
        # PROBLEM: must be returned to frontend and stored in the database
        
        self.risk_return_scatter_points = [
            {
                "ticker": ticker,
                "volatility": float(vol),
                "expectedReturn": float(expec_ret)
            }
            
            for ticker, vol, expec_ret in zip(tickers, self.assets_volatilities, annualized_mean_returns)
        ] 
        self.ef = EfficientFrontier(annualized_mean_returns, covarience_matrix,verbose=False,solver='CLARABEL')# possible solvers : ['CLARABEL', 'HIGHS', 'OSQP', 'SCIP', 'SCIPY', 'SCS']
        #self.ef = EfficientFrontier(
          #  bl_posterior_rets, 
           # bl_posterior_cov, 
            #weight_bounds=(0.0, 0.15), 
            #solver='CLARABEL'
        #)
        ###################
        self.efficient_frontier_points=compute_frontier(self.ef)       #PROBLEM: STORE THIS VALUE IN THE DATABASE.                                 
        
        #Optimization
        capital_allocations_percentages:OrderedDict[int,float]=self.get_allocations_percentages()
        
        print("309 good")
        # drop assets that have no allocation .
        capital_allocations_percentages={
            asset_name:capital_allocation_percentage
            for asset_name, capital_allocation_percentage in capital_allocations_percentages.items() if capital_allocations_percentages[asset_name]>0
        }
        print("313  good")
        performence_metrics=self.ef.portfolio_performance(verbose=False)                                         
        print("318  good")
        performence_metrics=tuple(map(float, performence_metrics))
        print("320  good")
        parsed_performance_metrics={
            "expectedAnnualReturn": performence_metrics[0],
            "annualVolatility": performence_metrics[1],   
            "sharpeRatio": performence_metrics[2]
            }
        print("322  good")
        latest_prices =self.cleaned_prices.xs("Close",level=1,axis=1).iloc[-1]
        print("328  good")
        
        da = DiscreteAllocation(self.ef.clean_weights(), latest_prices, total_portfolio_value=self.total_portfolio_value)
        shares_quantities, leftover = da.lp_portfolio(verbose=False) # Problem :  reinvest is related to the rebalancing
        self.leftover=float(leftover)
        print("333  good")
        #PROBLEM: some allocations are lost, those allocations must be added to the leftover for accruacy, so that the user does not get confused on where his money gone !
        capital_allocations_percentages={k:v for k,v in capital_allocations_percentages.items() if k in list(shares_quantities.keys()) }
        print("336  good") #this line was executed, so every thing above is correct
        self.optimal_portfolio:OptimalPortfolio={
        
            "assets":[
                {
                    "assetName":asset_name,
                    "capitalAllocationPercentage":capital_allocations_percentages[asset_name],
                    "quantity":shares_quantities[asset_name]
                }
                for asset_name in capital_allocations_percentages.keys()
            ] ,
            
            
            "metrics":parsed_performance_metrics
        }
        print("338  good")# this line was not executed, so, something starting at line 338 is wrong

    # the following five methods are related.    
    def get_avg_spreads(self,tickers:list[str])->dict[str,float]:
        result={}
        
        for ticker in tickers:
            ticker_df = self.cleaned_prices[ticker].copy()
            result[ticker] = ba.edge(ticker_df['Open'], ticker_df['High'], 
                                ticker_df['Low'], ticker_df['Close'])
        print('line 358 has no problem')
        return result
    
    def get_shares_outstanding(self,tickers)->dict[str,int]:
        shares = {}
        
        for t in tickers:
            try:
                fi = yf.Ticker(t).fast_info
                s = fi.get('shares')
                
                # fallback if missing
                if s is None:
                    s = yf.Ticker(t).info.get('sharesOutstanding')
                    
                shares[t] = s
                
            except Exception:
                shares[t] = None
                
        return shares

    def calc_slope(self,price_series):
        """
        Calculates the linear trend (slope) of the price series.
        Prices are normalized to a starting value of 1 so the slope represents
        an average period-over-period percentage drift.
        """
        prices = price_series.dropna()
        if len(prices) < 2:
            return 0.0
        
        # Normalize to evaluate relative trend
        y = prices.values / prices.values[0]
        x = np.arange(len(y))
        
        # Linear regression to find the slope (degree 1 polynomial)
        slope, _ = np.polyfit(x, y, 1)
        return slope

    def equities_filtering_algorithm(self,top_n, adv_threshold:int,slop_threshold:float=0.0): #PROBLEM : Add additional filtering layer based on rolling sharp ratio, not just slope,  or delete slope entirely.
        equities_close_prices=self.equities_prices.xs('Close',axis=1,level=1)
        print('line 400 has no problem')
        
        spreads = pd.Series(self.get_avg_spreads(list(equities_close_prices.columns)))
        common_index = self.equities_market_caps.index.intersection( self.average_daily_volume.index).intersection(spreads.index)
        
        rank_cap = self.equities_market_caps.loc[common_index].rank(ascending=False)
        rank_vol =  self.average_daily_volume.loc[common_index].rank(ascending=False)
        rank_spread = spreads.loc[common_index].rank(ascending=True)  # less spread → more liquidity → give less spread  less rank.
        
        combined_ranks = rank_cap + rank_vol+rank_spread
        U0 = combined_ranks.nsmallest(top_n).index.tolist()
        
        final_U = []
        
        # Step B: Iterate through the top N assets
        for sym in U0:
            if sym not in equities_close_prices.columns:# here
                continue
                
            equity_close_price = equities_close_prices[sym]#here
            
            # Calculate slope
            slope = self.calc_slope(equity_close_price)
            
            # Calculate daily log returns and their standard deviation (volatility)
            log_returns = np.log(equity_close_price / equity_close_price.shift(1)).dropna()
            
            
            # Assign labels based on computed metrics
            if slope >= slop_threshold:
                label = "up"
            elif slope <= -slop_threshold:
                label = "down"   
            else:
                label = "sideways"
                
            # Step C: Final Selection
            # Keep symbols labeled "up" or "volatile" AND verify high liquidity
            is_high_liquidity =  self.average_daily_volume.get(sym, 0) >= adv_threshold
            
            if label in ["up"] and is_high_liquidity:
                final_U.append(sym)
                
        return final_U

    def select_assets(self)->None:

        #BASE_DIR = Path(__file__).resolve().parent
        #file_path = BASE_DIR / "class_mapper.txt"
        with open('class_mapper.txt', 'r') as f:
            stored_dict = json.load(f)
        initial_equities_tickers=[key for key in list(stored_dict.keys()) if stored_dict[key]=="Equities"]
        first_service=HistoricalPricesService(ApiOrMockPricesData(assets_tickers=initial_equities_tickers, start_date="2024-1-1",interval="1d" )) #PROBLEM:the definition of the HistoricalPricesService class is outside the main class , is it ok or not ? can I put definition of class inside class ?
        
        
        """ 
        #other way to get 'first service'
        BASE_DIR = Path(__file__).resolve().parent
        first_service=HistoricalPricesService(MockData(BASE_DIR / "cleaned_multiindex_prices.parquet")) 
        """
        
        
        _,self.cleaned_prices=first_service.get_data()#this  method call never changes regardless of the underlying implementation ; DIP design pattern.
        survived_tickers = self.cleaned_prices.columns.get_level_values(0).unique().tolist()
        
        survived_equities=[equity for equity in  initial_equities_tickers if equity in survived_tickers ]
        # all OHLC prices of all equities; not just Close prices
        self.equities_prices=self.cleaned_prices[survived_equities]

        # get market capitalization
        equities_num_shares = pd.Series(self.get_shares_outstanding(survived_equities))
        
        equities_latest_close_prices = self.equities_prices.xs('Close', level=1, axis=1).iloc[-2] # don't use -1
        print("line 470 has no problem")
        self.equities_market_caps= (equities_latest_close_prices*equities_num_shares).dropna()
        
        self.average_daily_volume = self.equities_prices.xs('Volume', level=1, axis=1).iloc[-50:].mean()                    
        
        selected_equities = self.equities_filtering_algorithm(top_n=int(0.2 * len(survived_equities)),adv_threshold=1_000_000,slop_threshold=0.001 )

        #filter commodities
        #initial_commodities_tickers=[key for key in list(stored_dict.keys()) if stored_dict[key]=="Commodity"]
        #survived_commodities=[commodity for commodity in  initial_commodities_tickers if commodity in survived_tickers]
        #selected_commodities=filter(survived_commodities) # different filtering logic is needed based on industry and academic standards. we must study the concepts related to filtering commodities assests, so we need to study what are the metrics or features that determines filtering logic.

        #filter Fixed Income
        #initial_fixedincome_tickers=[key for key in list(stored_dict.keys()) if stored_dict[key]=="Fixed Income"]
        #survived_fixedincome=[fixedincome for fixedincome in initial_fixedincome_tickers if fixedincome in survived_tickers]
        #selected_fixedincome=filter(survived_fixedincome) # different filtering logic is needed based on industry and academic standards. we must study the concepts related to filtering fixed income assests, so we need to study what are the metrics or features that determines filtering logic.

        self.selected_assets:list[str]=selected_equities #+# selected_commodities #+# selected_fixedincome
        self.cleaned_prices=self.cleaned_prices[self.selected_assets]
    #standalone method
    def get_risk_appetite(self)->None:
        
        total_weight = sum(self.answers_weights)
        total_weighted_score = 0    
        for i in range(len(self.questions_scores)):
            total_weighted_score += self.questions_scores[i]*self.answers_weights[i]
            

        """if total_weight == 0:
            return "Unable to determine risk appetite"  # Handle edge case"""

        average_score = total_weighted_score / total_weight
        max_volatility = 1
        self.normalized_score = (average_score / 10) * max_volatility 
        if self.normalized_score <= 0.33333333: 
            self.risk_appetite = "Conservative"
        
        elif 0.33333334 <= self.normalized_score <= 0.66666666:
            self.risk_appetite = "Moderate"
        
        else:
            self.risk_appetite = "Aggressive"

    def get_investement_advice(self)->InvestementsAdviceMocks:
        self.get_risk_appetite()
        self.select_assets()
        self.perform_assets_allocation()
        print(self.selected_assets)
        return InvestementsAdviceMocks(
                leftover=self.leftover,
                optimalPortfolio=self.optimal_portfolio,
                assetsScatter=self.risk_return_scatter_points,
                efficientFrontierPoints=self.efficient_frontier_points,
            )



In [5]:
import json
with open("class_mapper.txt",'r') as f:
    stored_dict=json.load(f)
equities=list(stored_dict.keys())
x=yf.download(equities,start="2020-04-09",auto_adjust=True,threads=True,interval="1d",group_by="ticker")
close_returns=x.xs("Close",level=1,axis=1).pct_change().dropna()
df=close_returns.to_parquet("latest_close_returns.parquet")


[*********************100%***********************]  484 of 484 completed


DETECTING DISTRIBUTION OF THE VOLITILITIES

In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow import keras
import matplotlib.pyplot as plt

from sklearn import preprocessing
from tensorflow.keras import regularizers
#getting data
df = pd.read_parquet("latest_close_returns.parquet")

#deriving label
def create_sequences(data:pd.DataFrame, window:int)->np.typing.NDArray:
    """data is just time series, we need to make labels

    Args:
        data (_type_): _description_
        window (_type_): _description_

    Returns:
        _type_: I want specify output type hint ?
    """
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data.iloc[i:i + window])
        y.append(data.iloc[i + window,:])
    return np.array(X), np.array(y)
X, y = create_sequences(df, 100) # play with window

#splitting; not using train_test_split to avoid shuffling.
split = int(0.85 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

#Preprocessing : Scaling. scaling needs 2D data, not 3D data.
#After scaling is done, reshape again to 3D, as training need 3D data.
X_train_2D = X_train.reshape(-1, X_train.shape[2])
X_test_2D = X_test.reshape(-1, X_test.shape[2])
scaler=preprocessing.RobustScaler() # play with scaler
y_scaler = preprocessing.RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_2D).reshape(X_train.shape) # fit_tranform only once in training data. transform on the rest of the data
X_test_scaled = scaler.transform(X_test_2D).reshape(X_test.shape)
y_train_scaled = y_scaler.fit_transform(y_train)

#validation data (20 percent of the training data)
val_split = int(0.8 * len(X_train))
X_tr, X_val = X_train_scaled[:val_split], X_train_scaled[val_split:]
y_tr, y_val = y_train_scaled[:val_split], y_train_scaled[val_split:]



#model definition
model = keras.Sequential()
model.add(keras.layers.LSTM(256,return_sequences=True,kernel_regularizer=regularizers.l2(0.0005),input_shape=(#updated
    X_tr.shape[1], X_tr.shape[2])))
model.add(keras.layers.Dropout(0.2))
model.add(keras.layers.LSTM(512,kernel_regularizer=regularizers.l2(0.0005)))#updated
#model.add(keras.layers.LSTM(100))
model.add(keras.layers.Dropout(0.3))#updated
model.add(keras.layers.Dense(484))
model.compile(
        loss=keras.losses.Huber(delta=1.0), #updated
        optimizer=keras.optimizers.Adam(learning_rate=1e-4), #updated
        metrics=["mae"]#updated
    )



#model_trianing
callback=keras.callbacks.EarlyStopping(
    monitor="val_loss",
    min_delta=0,
    patience=10,
    verbose=0,
    mode="min",
    baseline=None,
    restore_best_weights=True,
    start_from_epoch=0,
)
history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=150,
    callbacks=[callback],
    batch_size=128 # updated
)
# 8. Plotting Learning Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.plot(history.history["loss"], label="Train Loss (Huber)")
ax1.plot(history.history["val_loss"], label="Validation Loss (Huber)")
ax1.set_title("Learning Curve (Loss)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid()

ax2.plot(history.history["mae"], label="Train MAE")
ax2.plot(history.history["val_mae"], label="Validation MAE")
ax2.set_title("Learning Curve (MAE)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("MAE")
ax2.legend()
ax2.grid()

plt.tight_layout()
plt.show()

# 9. Evaluation
pred_scaled = model.predict(X_test_scaled)
pred = y_scaler.inverse_transform(pred_scaled)    

mse = mean_squared_error(y_test, pred)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

# 10. Plotting Predictions
time = df.index[-len(y_test):]
plt.figure(figsize=(12, 6))

# Plotting actuals
plt.plot(time, y_test[:, 0], label="Actual", alpha=0.7)

# Plotting predictions (should now show variance instead of a flat line)
plt.plot(time, pred[:, 0], linestyle="--", linewidth=2, label="Predicted")

highlight = int(len(time) * 0.85)
if highlight < len(time):
    plt.axvspan(time[highlight], time[-1], alpha=0.1, color='gray')
    
plt.title("Asset Return Predictions vs Ground Truth (Asset 0)")
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"MSE : {mse:.6f}")
print(f"MAE : {mae:.6f}")
print(f"R2  : {r2:.6f}")

                              

I0000 00:00:1777561201.762982    2426 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777561227.848749    2426 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2857 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1050 Ti, pci bus id: 0000:01:00.0, compute capability: 6.1
/home/abd/myWorkSpace/FinSight_AI/backend/venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/150


I0000 00:00:1777561234.817887    2871 cuda_dnn.cc:461] Loaded cuDNN version 91002


8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 229ms/step - loss: 0.8861 - mae: 0.6764 - val_loss: 0.8724 - val_mae: 0.6667
Epoch 2/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 184ms/step - loss: 0.8645 - mae: 0.6756 - val_loss: 0.8515 - val_mae: 0.6665
Epoch 3/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 175ms/step - loss: 0.8434 - mae: 0.6749 - val_loss: 0.8311 - val_mae: 0.6664
Epoch 4/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 168ms/step - loss: 0.8230 - mae: 0.6744 - val_loss: 0.8114 - val_mae: 0.6662
Epoch 5/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 169ms/step - loss: 0.8033 - mae: 0.6739 - val_loss: 0.7923 - val_mae: 0.6661
Epoch 6/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 180ms/step - loss: 0.7842 - mae: 0.6733 - val_loss: 0.7740 - val_mae: 0.6660
Epoch 7/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 183ms/step - loss: 0.7659 - mae: 0.6729 - val_loss: 0.7563 - val_mae: 0.6659
Epoch 8/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 181ms/step - loss: 0.7482 - mae: 0.6725 - val_loss: 0.7393 - val_mae: 0.6658
Epoch 9/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 180ms/step - loss: 0.7312 - mae: 0.6

In [8]:
pred.mean()

np.float32(0.0012098072)

In [ ]:
import pandas as pd
import numpy as np
from pypfopt.expected_returns import mean_historical_return
from pypfopt.discrete_allocation import DiscreteAllocation
from collections import OrderedDict
from pandas import DataFrame
from pandas import Series
from typing import Any
from numpy.typing import NDArray
from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.efficient_frontier import EfficientFrontier
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
from pydantic import BaseModel
from typing import TypedDict
import yfinance as yf

import time
import json
from abc import ABC, abstractmethod

import bidask as ba
import json
from pathlib import Path
from pathlib import Path



ModuleNotFoundError: No module named 'app'